# Bronze ingestion test

Ingests `data/B001/customer_B001.csv` into a bronze table using the `pipeline` functions directly. The cells show:
- a first load (all rows inserted)
- a rerun of the same file (idempotency: every row is a replay, nothing inserted)
- a failed run, with the error naming the step

In [1]:
import sys
from dataclasses import asdict
from datetime import datetime, timezone
from pathlib import Path

# Make src/ importable when the notebook runs from anywhere inside the project
root = Path.cwd()
while not (root / 'src').exists() and root != root.parent:
    root = root.parent
sys.path.insert(0, str(root / 'src'))

from pyspark.sql.types import StructType, StructField, StringType
from pipeline.session import get_spark_session
from pipeline.catalog import ensure_schema_exists, get_schema_fqn, get_table_fqn
from pipeline.audit import AuditRun
from pipeline.errors import step
from pipeline.metadata import add_metadata
from pipeline.reader import read_source_csv
from pipeline.validation import assert_columns_exist, count_duplicate_keys, count_null_business_keys
from pipeline.writer import insert_new_rows
from bronze.entity import EntityConfig

spark = get_spark_session('bronze_ingestion_test')

Picked up JAVA_TOOL_OPTIONS: -Djavax.net.ssl.trustStore=/home/charlescarpenter/.local/share/Project_Mercury/cacerts -Djavax.net.ssl.trustStorePassword=changeit
Picked up JAVA_TOOL_OPTIONS: -Djavax.net.ssl.trustStore=/home/charlescarpenter/.local/share/Project_Mercury/cacerts -Djavax.net.ssl.trustStorePassword=changeit
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/10/11 15:12:31 WARN Utils: Your hostname, DP41ZL4, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/10/11 15:12:31 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/home/charlescarpenter/venvs/mercury-delta/lib/python3.14/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/charlescarpenter/.ivy2.5.2/cache
The jars for the packages stored in: /home/charlescarpenter/.ivy2.5.2/jars
io.delta#delta-spark_4.1_2.13 added as a dep

In [2]:
import logging
from pipeline.events import EventBus
from pipeline.log import make_log_subscriber

logging.basicConfig(format='%(asctime)s %(levelname)s %(message)s', level=logging.INFO)

bus = EventBus()
bus.subscribe(make_log_subscriber(logging.getLogger('pipeline')))

1

In [3]:
ENTITY_FILES = [
    'address', 'category', 'customer', 'order', 'order_item', 'payment',
    'product', 'product_category', 'return', 'return_item',
]

PROJECT_DATA = root / 'data' / 'B001'
CATALOG = 'forge'
TEST_SCHEMA = 'bronze'
ensure_schema_exists(spark, CATALOG, TEST_SCHEMA)

def build_entity(name: str) -> EntityConfig:
    source = PROJECT_DATA / f'{name}_B001.csv'
    header = source.read_text().splitlines()[0].split(',')
    # Bronze keeps every source column as a string
    schema = StructType([StructField(c, StringType(), True) for c in header])
    return EntityConfig(
        name=name,
        source_path=str(source),
        schema=schema,
        target_table=get_table_fqn(CATALOG, TEST_SCHEMA, name),
        merge_keys=(f'{name}_id',),
    )

ENTITIES = [build_entity(name) for name in ENTITY_FILES]
for e in ENTITIES:
    print(f'{e.name:18} {len(e.schema.fields):3} columns -> {e.target_table}')

address             13 columns -> bronze.address
category             8 columns -> bronze.category
customer            11 columns -> bronze.customer
order               16 columns -> bronze.order
order_item          11 columns -> bronze.order_item
payment             12 columns -> bronze.payment
product             10 columns -> bronze.product
product_category     9 columns -> bronze.product_category
return              10 columns -> bronze.return
return_item         10 columns -> bronze.return_item


In [4]:
records = []  # the audit sink: AuditRun appends each finished AuditRecord here

def ingest(entity: EntityConfig):
    with AuditRun('bronze', entity.name, entity.target_table, sink=records.append, bus=bus) as run:
        with step('read', bus=bus, path=entity.source_path):
            df = read_source_csv(spark, entity.schema, entity.source_path, entity.read_options, file_path_column='_source_file')

        with step('add_metadata', bus=bus):
            df = add_metadata(
                df,
                run_id=run.record.run_id,
                ingested_at=datetime.now(timezone.utc),
                source_columns=[f.name for f in entity.schema.fields],
            )

        with step('validate', bus=bus):
            assert_columns_exist(df, entity.merge_keys)
            run.add('source_row_count', df.count())
            run.add('null_business_key_count', count_null_business_keys(df, entity.merge_keys))
            run.add('duplicate_business_key_count', count_duplicate_keys(df, entity.merge_keys))

        with step('write', table=entity.target_table, bus=bus):
            result = insert_new_rows(spark, df, entity.target_table, entity.merge_keys, hash_column='_record_hash')
            for name, value in asdict(result).items():
                run.add(name, value)

    return records[-1]

def show(record):
    print(record.entity, record.status, record.metrics, record.error_message or '')

In [5]:
# First load: every entity, every row should be inserted
results = {e.name: ingest(e) for e in ENTITIES}
for name, record in results.items():
    print(f"{name:18} {record.status:8} inserted={record.metrics.get('inserted_row_count')}")

2026-10-11 15:12:38,907 INFO bronze address run f60936f1-ea99-43d5-8d4b-5dc42941926b started
2026-10-11 15:12:39,208 INFO step 'read' (path=/home/charlescarpenter/python/spark_local/data_pipline_v2/data/B001/address_B001.csv) completed in 0.30s
2026-10-11 15:12:39,366 INFO step 'add_metadata' completed in 0.16s
2026-10-11 15:12:41,435 INFO step 'validate' completed in 2.07s
2026-10-11 15:12:47,493 INFO step 'write' (table=bronze.address) completed in 6.06s
2026-10-11 15:12:47,495 INFO bronze address run f60936f1-ea99-43d5-8d4b-5dc42941926b succeeded: {'source_row_count': 68, 'null_business_key_count': 0, 'duplicate_business_key_count': 0, 'replay_existing_row_count': 0, 'inserted_row_count': 68, 'hash_conflict_count': 0, 'target_total_row_count': 68}
2026-10-11 15:12:47,496 INFO bronze category run d7626ea7-4512-4932-aa42-63067c18fe0f started
2026-10-11 15:12:47,533 INFO step 'read' (path=/home/charlescarpenter/python/spark_local/data_pipline_v2/data/B001/category_B001.csv) completed i

address            success  inserted=68
category           success  inserted=6
customer           success  inserted=50
order              success  inserted=1000
order_item         success  inserted=3096
payment            success  inserted=1036
product            success  inserted=200
product_category   success  inserted=310
return             success  inserted=80
return_item        success  inserted=142


In [6]:
for entity, record in results.items():
    print(entity, record.metrics['inserted_row_count'])

address 68
category 6
customer 50
order 1000
order_item 3096
payment 1036
product 200
product_category 310
return 80
return_item 142


In [7]:
# Rerun of every file: every row is a replay, so nothing is inserted
results = {e.name: ingest(e) for e in ENTITIES}
for name, record in results.items():
    print(f"{name:18} {record.status:8} replayed={record.metrics.get('replay_existing_row_count')}")

2026-10-11 15:13:02,231 INFO bronze address run 5cac84b5-abf5-4516-b082-1e54e5b58391 started
2026-10-11 15:13:02,283 INFO step 'read' (path=/home/charlescarpenter/python/spark_local/data_pipline_v2/data/B001/address_B001.csv) completed in 0.05s
2026-10-11 15:13:02,337 INFO step 'add_metadata' completed in 0.05s
2026-10-11 15:13:02,627 INFO step 'validate' completed in 0.29s
2026-10-11 15:13:05,332 INFO step 'write' (table=bronze.address) completed in 2.70s
2026-10-11 15:13:05,334 INFO bronze address run 5cac84b5-abf5-4516-b082-1e54e5b58391 succeeded: {'source_row_count': 68, 'null_business_key_count': 0, 'duplicate_business_key_count': 0, 'replay_existing_row_count': 68, 'inserted_row_count': 0, 'hash_conflict_count': 0, 'target_total_row_count': 68}
2026-10-11 15:13:05,334 INFO bronze category run b61feade-998f-45ed-bdbe-47a2a1cdf670 started
2026-10-11 15:13:05,367 INFO step 'read' (path=/home/charlescarpenter/python/spark_local/data_pipline_v2/data/B001/category_B001.csv) completed i

address            success  replayed=68
category           success  replayed=6
customer           success  replayed=50
order              success  replayed=1000
order_item         success  replayed=3096
payment            success  replayed=1036
product            success  replayed=200
product_category   success  replayed=310
return             success  replayed=80
return_item        success  replayed=142


In [8]:
# Row count per bronze table
for e in ENTITIES:
    print(f'{e.name:18} {spark.read.table(e.target_table).count()}')

address            68
category           6
customer           50
order              1000
order_item         3096
payment            1036
product            200
product_category   310
return             80
return_item        142


In [9]:
# A missing source file: expect status 'failed' and an error naming the 'read' step
from dataclasses import replace

missing = replace(ENTITIES[0], source_path=str(PROJECT_DATA / 'missing.csv'))
try:
    ingest(missing)
except Exception as exc:
    print('raised:', exc)

show(records[-1])

2026-10-11 15:13:22,743 INFO bronze address run 7ea28d49-1bfb-4d35-9887-3ec3b238676c started
2026-10-11 15:13:22,820 ERROR step 'read' (path=/home/charlescarpenter/python/spark_local/data_pipline_v2/data/B001/missing.csv) failed: AnalysisException: [PATH_NOT_FOUND] Path does not exist: file:/home/charlescarpenter/python/spark_local/data_pipline_v2/data/B001/missing.csv. SQLSTATE: 42K03
2026-10-11 15:13:22,820 ERROR bronze address run 7ea28d49-1bfb-4d35-9887-3ec3b238676c failed: Step 'read' failed (path=/home/charlescarpenter/python/spark_local/data_pipline_v2/data/B001/missing.csv): AnalysisException: [PATH_NOT_FOUND] Path does not exist: file:/home/charlescarpenter/python/spark_local/data_pipline_v2/data/B001/missing.csv. SQLSTATE: 42K03


raised: Step 'read' failed (path=/home/charlescarpenter/python/spark_local/data_pipline_v2/data/B001/missing.csv): AnalysisException: [PATH_NOT_FOUND] Path does not exist: file:/home/charlescarpenter/python/spark_local/data_pipline_v2/data/B001/missing.csv. SQLSTATE: 42K03
address failed {} Step 'read' failed (path=/home/charlescarpenter/python/spark_local/data_pipline_v2/data/B001/missing.csv): AnalysisException: [PATH_NOT_FOUND] Path does not exist: file:/home/charlescarpenter/python/spark_local/data_pipline_v2/data/B001/missing.csv. SQLSTATE: 42K03


In [10]:
for entity, record in results.items():
    print(entity, record.metrics['inserted_row_count'])

address 0
category 0
customer 0
order 0
order_item 0
payment 0
product 0
product_category 0
return 0
return_item 0


In [ ]:
# Teardown is in notebook 00; uncomment to drop the bronze schema here instead
# spark.sql(f'DROP SCHEMA IF EXISTS {get_schema_fqn(CATALOG, TEST_SCHEMA)} CASCADE')
